## Phase 4 — Report (Steps 22–23)

Define KPIs and charts for the final dashboard.

### Step 22 — Identify Key Metrics / KPIs for Dashboard
Chosen based on the problem statement (Step 12) and the Operations/Sales domain — each is
measurable, tied to the problem statement, and drives an actual decision:

- **Total Revenue** (`net_revenue_inr` summed)
- **Total Trips** and **Completed Trips**
- **Cancellation Rate %** (Cancelled + No-show ÷ total)
- **On-Time Performance %** (`delay_minutes` ≤ 5)
- **Average Fare** and **Average Occupancy %**
- **Average Rating**
- **Complaint Rate %**
- **Revenue by City** (identifies top/bottom performing metros)

In [ ]:
kpis = {
    "Total Revenue (INR)": df["net_revenue_inr"].sum(),
    "Total Trips": len(df),
    "Completed Trips": (df["trip_status"].isin(["Completed", "Delayed-Completed"])).sum(),
    "Cancellation Rate %": (df["trip_status"].isin(["Cancelled", "No-show"]).mean() * 100),
    "On-Time Performance %": ((df["delay_minutes"] <= 5).mean() * 100),
    "Average Fare (INR)": df["fare_inr"].mean(),
    "Average Occupancy %": df["occupancy_pct"].mean(),
    "Average Rating": df["rating"].mean(),
    "Complaint Rate %": (df["complaint_raised"] == True).mean() * 100,
}
kpi_df = pd.DataFrame.from_dict(kpis, orient="index", columns=["value"]).round(2)
kpi_df

In [ ]:
df.groupby("city")["net_revenue_inr"].sum().sort_values(ascending=False).round(0)

### Step 23 — Identify Important Charts for Dashboard
Chart types recommended by the checklist, applied to this dataset:

- **KPI cards** — the headline metrics from Step 22
- **Trend line chart** — revenue/trips over time (month)
- **Bar chart** — revenue by city
- **Pie/donut chart** — trip status split (already shown in Step 18)
- **Heatmap** — correlation matrix (already shown in Step 20) / weekday × city trip-density
- **Geo map** — not directly available (no lat/long in the data); city-level bar chart substitutes
- **Funnel chart** — booking → completed trip funnel
- **Scatter plot** — fare vs distance (already shown in Step 19)

In [ ]:
monthly = df.groupby(df["trip_date"].dt.to_period("M")).agg(
    trips=("trip_id", "count"), revenue=("net_revenue_inr", "sum")
)
monthly.index = monthly.index.astype(str)

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(monthly.index, monthly["revenue"], marker="o", color="#3b6fa0", label="Revenue (INR)")
ax1.set_ylabel("Revenue (INR)", color="#3b6fa0")
ax1.tick_params(axis="x", rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly["trips"], marker="s", color="#c96a3f", label="Trips")
ax2.set_ylabel("Trips", color="#c96a3f")

ax1.set_title("Monthly Revenue & Trip Volume Trend (2024)")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
df.groupby("city")["net_revenue_inr"].sum().sort_values(ascending=False).plot.bar(ax=ax, color="#3b6fa0")
ax.set_title("Total Revenue by City")
ax.set_ylabel("Revenue (INR)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
heat_data = df.pivot_table(index="trip_weekday", columns="city", values="trip_id", aggfunc="count")
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heat_data = heat_data.reindex(weekday_order)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(heat_data, cmap="Blues", annot=True, fmt=".0f", ax=ax)
ax.set_title("Trip Volume Heatmap — Weekday x City")
plt.show()

In [ ]:
funnel_stages = {
    "Booked": len(df),
    "Not Cancelled/No-show": (~df["trip_status"].isin(["Cancelled", "No-show"])).sum(),
    "Completed (on-time or delayed)": df["trip_status"].isin(["Completed", "Delayed-Completed"]).sum(),
    "Completed On-Time": ((df["trip_status"].isin(["Completed", "Delayed-Completed"])) &
                           (df["delay_minutes"] <= 5)).sum(),
}
funnel_df = pd.DataFrame({"stage": list(funnel_stages.keys()), "count": list(funnel_stages.values())})
funnel_df["pct_of_booked"] = (funnel_df["count"] / funnel_df["count"].iloc[0] * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(funnel_df["stage"], funnel_df["count"], color="#3b6fa0")
for i, (c, p) in enumerate(zip(funnel_df["count"], funnel_df["pct_of_booked"])):
    ax.text(c, i, f"  {c} ({p}%)", va="center")
ax.invert_yaxis()
ax.set_title("Booking -> Completion Funnel")
plt.tight_layout()
plt.show()
funnel_df

## Conclusion

All 23 steps of the checklist have been applied end-to-end: the raw dataset was inspected (Steps
1–8), cleaned and prepared into `cityflo_bus_service_metro_cities_cleaned.csv` (Steps 9–17),
statistically analyzed across univariate, bivariate, multivariate, and hypothesis-testing lenses
(Steps 18–21), and summarized into dashboard-ready KPIs and charts (Steps 22–23).

**Key findings to carry into a dashboard:**
- Fare scales strongly with distance (positive Pearson correlation).
- Bus type materially affects average fare (ANOVA significant).
- Delay behavior differs between peak and non-peak hours (t-test).
- Cancellation/no-show patterns are associated with city (Chi-square significant).
